This is the notebook for generating graphs

In [1]:
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt

Run the `tester.py` on the cloud machine with vitis first to get the `out/q?/_table.md` files. Then run this notebook on your local machine.
This is done because the cloud machines dont let me install packages like matplotlib or the ipynb kernel.

In [2]:
def plot_markdown_metrics(table_path):
    table_path = Path(table_path).resolve()
    parent_dir = table_path.parent
    
    rotations = []
    throughputs = []
    luts = []
    ffs = []
    dsps = []
    rmse_r = []
    rmse_theta = []

    with open(table_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line.startswith("|") or "Design" in line or "---" in line:
                continue
            parts = [p.strip() for p in line.split("|")[1:-1]]
            if len(parts) >= 10:
                try:
                    rotations.append(int(parts[0]))
                    throughputs.append(float(parts[7]))
                    luts.append(int(parts[5]))
                    ffs.append(int(parts[6]))
                    dsps.append(int(parts[4]))
                    rmse_r.append(float(parts[8]))
                    rmse_theta.append(float(parts[9]))
                except ValueError:
                    continue

    plots = [
        ("throughput", "Throughput vs. Rotations", "Throughput (MHz)", [
            (rotations, throughputs, "Throughput", "tab:blue", "o", "{:.2f}")
        ], False),
        ("resource_usage", "Resource Usage vs. Rotations", "Resource Count", [
            (rotations, luts, "LUT", "tab:orange", "o", "{}"),
            (rotations, ffs, "FF", "tab:green", "s", "{}"),
            (rotations, dsps, "DSP", "tab:red", "^", "{}")
        ], False),
        ("rmse", "RMSE vs. Rotations", "RMSE (Log Scale)", [
            (rotations, rmse_r, "RMSE (r)", "tab:purple", "o", "{:.2e}"),
            (rotations, rmse_theta, "RMSE (theta)", "tab:cyan", "s", "{:.2e}")
        ], True)
    ]

    for suffix, title, ylabel, series, log_scale in plots:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        for x_vals, y_vals, label, color, marker, fmt in series:
            ax.plot(x_vals, y_vals, label=label, color=color, marker=marker)
            for x, y in zip(x_vals, y_vals):
                ax.annotate(
                    fmt.format(y),
                    (x, y),
                    textcoords="offset points",
                    xytext=(0, 6),
                    ha="center",
                    fontsize=8
                )
        
        ax.set_title(title)
        ax.set_xlabel("Number of Rotations")
        ax.set_ylabel(ylabel)
        if log_scale:
            ax.set_yscale("log")
            ax.grid(True, which="both", ls="--")
        else:
            ax.grid(True)
        if len(series) > 1:
            ax.legend()
            
        plt.tight_layout()
        out_file = parent_dir / f"graph_{suffix}.png"
        plt.savefig(out_file, dpi=300)
        plt.close(fig)


Generate graph for Q1

In [3]:
base_dir = Path.cwd().parent / "project2" / "cordic" / "HLS" / "cordic" / "out"
plot_markdown_metrics(base_dir / "q1" / "_table.md")

@harini @avi, put your other questions with the path to the '_table.md'